# 전처리 2단계 — school_candidate 정규화 (`preprocessed_candidate` 컬럼)

이후 GT match를 위한 사전 작업 

`school_candidate`의 각 토큰을 정식 학교명 형태로 통일한다:
1. GT의 약어(공백으로 여러 개 있을 수 있음, 예: "이대")에 있으면 -> 그 행의 정식명("이화여자대학교")으로 바꿈
2. 아니면 접미사 확장(예: 서초중 -> 서초중학교)

GT에 실제로 없는 건(예: "먹고" -> "먹고등학교") 이 단계에서는 그냥 통과시키고,
`4.0-gt-match.ipynb`에서 GT 정식명 목록과 대조할 때 자연히 버려진다.

In [1]:
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "src").is_dir() and (p / "data").is_dir():
            return p
    raise FileNotFoundError("repo 루트를 못 찾았어요 (data/, src/ 폴더 기준)")


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT / "src" / "utils") not in sys.path:
    sys.path.append(str(REPO_ROOT / "src" / "utils"))

REPO_ROOT

WindowsPath('D:/Study/dongguk_university/dreampath')

In [2]:
import pandas as pd

df = pd.read_csv(
    REPO_ROOT / "data" / "interim" / "school_candidate_counts.csv", encoding="utf-8-sig"
)
df["school_candidate"] = df["school_candidate"].fillna("")  # 후보 0건인 행은 NaN으로 읽히므로 방어
df.shape

(1000, 6)

## school_candidate의 각 토큰을 접미사 확장 -> preprocessed_candidate

In [ ]:
from utils import preprocess_candidate

gt_df = pd.read_csv(REPO_ROOT / "data" / "processed" / "gt_schoolnames.csv", encoding="utf-8-sig")
# 약어 컬럼은 build_abbreviations.ipynb를 돌린 뒤에만 생김 (아직이면 빈 dict여도 정상)
# 한 행에 여러 별칭이 공백으로 들어있을 수 있으므로 각각 풀어서 매핑
alias_to_canonical = {}
if "약어" in gt_df.columns:
    for aliases, name in zip(gt_df["약어"], gt_df["학교명"]):
        if pd.notna(aliases):
            for alias in aliases.split():
                alias_to_canonical[alias] = name


df["preprocessed_candidate"] = df["school_candidate"].apply(
    lambda s: preprocess_candidate(s, alias_to_canonical)
)
df[["comment", "school_candidate", "preprocessed_candidate"]].head(20)

In [4]:
out_path = REPO_ROOT / "data" / "interim" / "preprocessed_school_candidate.csv"
df.to_csv(out_path, index=False, encoding="utf-8-sig")
out_path

WindowsPath('D:/Study/dongguk_university/dreampath/data/interim/preprocessed_school_candidate.csv')